imports

In [234]:
import torch
from itertools import product

method that checks if Tensor A in shape that can be expanded to match the shape of tensor B. expanding rules are as follows:
1) Dimensions are compared from right to left.
2) Two dimensions are compatible if they are equal or if one of them is 1.(else they are incompatible and expansion is not possible)
method returns if false if not expandable and true if it is and if it is return the target shape the Tensor A should be expanded to (which is the shape of B with 1s padded on the left if needed to match the number of dimensions)

In [235]:
def can_expand_as(A: torch.Tensor, B: torch.Tensor) :
    shapeA = list(A.shape)
    shapeB = list(B.shape)
    max_len = max(len(shapeA), len(shapeB)) # we need this to iterate over target dims
    for index in range(1, max_len + 1): # iterate from right to left
        dimA = shapeA[-index] if index <= len(shapeA) else 1
        dimB = shapeB[-index] if index <= len(shapeB) else 1
        if dimA != dimB and dimA != 1: # if dims are not equal and dimA is not 1 then expansion is not possible
            return False, None
    padded_shapeB = [1] * (max_len - len(shapeB)) + shapeB # pad shapeB with 1s on the left to match max_len
    return True, tuple(padded_shapeB)

brodcasting method that takes two tensors A and B and checks if we can find a common shape that both tensors can be expanded to:
uses same logic as expansion :
1) Dimensions are compared from right to left.
2) if dimensions are equal trivial
3) if dimensions are not equal but one of them is 1, the shape in that dimension can be expanded to other shape
4) if dimensions are not equal and neither of them is 1, the shapes are not broadcastable and we should raise an error
5) if one of the dimensions is 0 the result shape in that dimension should be 0

In [236]:
def broadcast_shapes(A: torch.Tensor, B: torch.Tensor):
    shapeA = list(A.shape)
    shapeB = list(B.shape)
    max_len = max(len(shapeA), len(shapeB))
    result_shape = []

    for index in range(1, max_len + 1): # same iterate right to left
        dimA = shapeA[-index] if index <= len(shapeA) else 1
        dimB = shapeB[-index] if index <= len(shapeB) else 1
        if dimA != dimB and dimA != 1 and dimB != 1: # if dims are not equal and neither is 1 then shapes are not broadcastable
            raise ValueError("Shapes are not broadcastable")
        dim = max(dimA, dimB) if dimA !=0 and dimB != 0 else 0  # special case for one is zero then it have to be zero
        result_shape.append(dim)

    result_shape.reverse()
    return tuple(result_shape)

expansion method takes two tensors A and B checks if A can be expanded as B using the method we worte up , if returned value was false then shapes are not expandable then we raise value error , else shapes are compatible for expansion
we create a tensor with expansion shape (B shape with 1 padding) and the same type as the tensor A ,
then we calculate possible ranges to iterate over  then we iterate and for each index we calculate corrsponding index in A to fill the value in tensor C

In [237]:
def expand_as_manually(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    ok, target_shape = can_expand_as(A, B)

    if not ok:
        raise ValueError("A cannot be expanded to match B")

    # pad A to match the dim shape
    padding_tensor = [1] * (len(target_shape) - len(A.shape))
    A_shape = padding_tensor + list(A.shape)

    # create the output tensor in mman 11 written that we can do it no need to do it inplace
    C = torch.empty(target_shape, dtype=A.dtype)
    # save ranges that we should iterate over to fill the values in C
    dim_ranges = [range(s) for s in target_shape]
    a_ndim = A.dim()

    for idx in product(*dim_ranges):
        src_idx_full = []
        for i in range(len(target_shape)):
            src_idx_full.append(0 if A_shape[i] == 1 else idx[i]) # most tricky part if in shape it's 1 then it needs we need the same value in A but the rest of the indcies should be zero since we can't iterate over the subset if it's not 1 then there is a place to iterate so we need the index in the original

        src_idx = tuple(src_idx_full[-a_ndim:]) if a_ndim > 0 else ()
        C[idx] = A[src_idx]

    return C

test cases for expansion method:
1) method1: _same() compares if two tensors have same tupe and are equal
2) method2: check() if shapes are expandable it compares both tensors using the _same method and updates counts of passed and failed tests and prints the result
3) method3: check_matches_torch() calls function with input tensors if it's expandable then expand and compare with torch.expand result if not expandable it should raise value error. if result is as expected then we increase passed count else we increase failed count and print the details of the failure

In [238]:
### Test cases for expand_as_manually
### if you want to add testcases call function check_matches_torch(name, A, B) with appropriate tensors A and B.

passed = 0
failed = 0

def _same(a: torch.Tensor, b: torch.Tensor) -> bool:
    return a.dtype == b.dtype and torch.equal(a, b)

def check(name, result, expected):
    global passed, failed
    if _same(result, expected):
        print(f"  PASS  {name}")
        passed += 1
    else:
        print(f"  FAIL  {name}")
        print(f"       got:      shape={tuple(result.shape)} dtype={result.dtype}\n{result}")
        print(f"       expected: shape={tuple(expected.shape)} dtype={expected.dtype}\n{expected}")
        failed += 1

def check_matches_torch(name, A: torch.Tensor, B: torch.Tensor):
    global passed, failed
    ok, target_shape = can_expand_as(A, B)

    if not ok:
        try:
            expand_as_manually(A, B)
            print(f"  FAIL  {name}  (expected ValueError, got success)")
            failed += 1
        except ValueError:
            print(f"  PASS  {name}  (correctly raised ValueError)")
            passed += 1
        return

    expected = A.expand(target_shape)
    result = expand_as_manually(A, B)
    check(name, result, expected)

print("=" * 60)
print("Edge-case tests for expand_as_manually (vs torch.expand)")
print("=" * 60)

# 1) Identical shapes (no expansion needed)
check_matches_torch("identical shapes (2,2)",
                    torch.tensor([[1, 2], [3, 4]]),
                    torch.zeros(2, 2))

# 2) Scalar (0-d) expands to any shape
check_matches_torch("scalar → (3,4)",
                    torch.tensor(7),
                    torch.zeros(3, 4))

# 3) 1-D expands to 2-D by adding leading dim
check_matches_torch("(3,) → (4,3)",
                    torch.tensor([1, 2, 3]),
                    torch.zeros(4, 3))

# 4) Column vector expands across columns
check_matches_torch("(3,1) → (3,4)",
                    torch.tensor([[1], [2], [3]]),
                    torch.zeros(3, 4))

# 5) Notebook example actually expands to (1,3,3) (target_shape from rules)
check_matches_torch("(1,1,3) vs (3,3) → target (1,3,3)",
                    torch.tensor([[[1, 2, 3]]]),
                    torch.tensor([[4, 5, 6],
                                  [7, 8, 9],
                                  [10, 11, 12]]))

# 6) Higher-rank broadcast
check_matches_torch("(1,3,1) → (2,3,4)",
                    torch.tensor([[[1], [2], [3]]]),
                    torch.zeros(2, 3, 4))

# 7) All-ones expands trivially
check_matches_torch("(1,1,1) → (5,5,5)",
                    torch.ones(1, 1, 1),
                    torch.zeros(5, 5, 5))

# 8) dtype preserved
A = torch.tensor([[1.5, 2.5]], dtype=torch.float32)
B = torch.zeros(3, 2)
res = expand_as_manually(A, B)
exp = A.expand(can_expand_as(A, B)[1])
check("dtype preserved (float32)", res, exp)

# 9) Empty dimension expansion (size 0)
check_matches_torch("(0,1) → (0,5)",
                    torch.empty(0, 1),
                    torch.empty(0, 5))

# 10) Non-contiguous input tensor should still work (indexing handles it)
base = torch.arange(12).reshape(3, 4)
A_nc = base.t()[:1, :]   # non-contiguous view, shape (1,3)
B = torch.zeros(6, 3)
check_matches_torch("non-contiguous (1,3) → (6,3)", A_nc, B)

# 11) Incompatible shapes should raise
check_matches_torch("incompatible (2,3) vs (2,4) raises",
                    torch.zeros(2, 3),
                    torch.zeros(2, 4))

check_matches_torch("incompatible (3,1) vs (4,2) raises",
                    torch.zeros(3, 1),
                    torch.zeros(4, 2))

print("=" * 60)
print(f"Results: {passed} passed, {failed} failed out of {passed+failed} tests")
if failed:
    raise AssertionError(f"{failed} test(s) failed")

Edge-case tests for expand_as_manually (vs torch.expand)
  PASS  identical shapes (2,2)
  PASS  scalar → (3,4)
  PASS  (3,) → (4,3)
  PASS  (3,1) → (3,4)
  PASS  (1,1,3) vs (3,3) → target (1,3,3)
  PASS  (1,3,1) → (2,3,4)
  PASS  (1,1,1) → (5,5,5)
  PASS  dtype preserved (float32)
  PASS  (0,1) → (0,5)
  PASS  non-contiguous (1,3) → (6,3)
  PASS  incompatible (2,3) vs (2,4) raises  (correctly raised ValueError)
  PASS  incompatible (3,1) vs (4,2) raises  (correctly raised ValueError)
Results: 12 passed, 0 failed out of 12 tests


test cases for broadcast_shapes method:
1) method1: check_b() compares the result of broadcast_shapes to the expected shape and updates counts of passed and failed tests and prints the result
2) method2: check_broadcast_matches_torch() compares the result of broadcast_shapes to the expected shape from torch.broadcast_tensors. If torch.broadcast_tensors raises, broadcast_shapes should raise too. If the result matches expected shape, we increase passed count else we increase failed count and print the details of the failure

In [239]:
### Tests for broadcast_shapes

passed_b = 0
failed_b = 0

def check_b(name, result, expected):
    global passed_b, failed_b
    if result == expected:
        print(f"  PASS  {name}")
        passed_b += 1
    else:
        print(f"  FAIL  {name}")
        print(f"       got:      {result}")
        print(f"       expected: {expected}")
        failed_b += 1

def check_broadcast_matches_torch(name, shapeA, shapeB):
    global passed_b, failed_b
    A = torch.empty(shapeA)
    B = torch.empty(shapeB)

    try:
        exp_shape = torch.broadcast_tensors(A, B)[0].shape
    except RuntimeError:
        try:
            broadcast_shapes(A, B)
            print(f"  FAIL  {name}  (expected ValueError, got success)")
            failed_b += 1
        except ValueError:
            print(f"  PASS  {name}  (correctly raised ValueError)")
            passed_b += 1
        return

    got = broadcast_shapes(A, B)
    check_b(name, got, tuple(exp_shape))

print("=" * 60)
print("Edge-case tests for broadcast_shapes (vs torch.broadcast_tensors)")
print("=" * 60)

# Basic / identical
check_broadcast_matches_torch("identical (2,3) & (2,3)", (2, 3), (2, 3))

# Scalar cases
check_broadcast_matches_torch("scalar () & (3,4)", (), (3, 4))
check_broadcast_matches_torch("scalar () & scalar ()", (), ())

# Leading dims alignment
check_broadcast_matches_torch("(3,) & (4,3)", (3,), (4, 3))
check_broadcast_matches_torch("(1,3,1) & (2,1,4)", (1, 3, 1), (2, 1, 4))

# Ones broadcast
check_broadcast_matches_torch("(1,1,1) & (5,5,5)", (1, 1, 1), (5, 5, 5))

# Empty dims (size 0)
check_broadcast_matches_torch("(0,1) & (0,5)", (0, 1), (0, 5))
check_broadcast_matches_torch("(0,1) & (1,5)", (0, 1), (1, 5))

# Non-broadcastable
check_broadcast_matches_torch("incompatible (2,3) & (2,4)", (2, 3), (2, 4))
check_broadcast_matches_torch("incompatible (3,1) & (4,2)", (3, 1), (4, 2))

print("=" * 60)
print(f"broadcast_shapes: {passed_b} passed, {failed_b} failed out of {passed_b+failed_b}")
if failed_b:
    raise AssertionError(f"broadcast_shapes: {failed_b} test(s) failed")


Edge-case tests for broadcast_shapes (vs torch.broadcast_tensors)
  PASS  identical (2,3) & (2,3)
  PASS  scalar () & (3,4)
  PASS  scalar () & scalar ()
  PASS  (3,) & (4,3)
  PASS  (1,3,1) & (2,1,4)
  PASS  (1,1,1) & (5,5,5)
  PASS  (0,1) & (0,5)
  PASS  (0,1) & (1,5)
  PASS  incompatible (2,3) & (2,4)  (correctly raised ValueError)
  PASS  incompatible (3,1) & (4,2)  (correctly raised ValueError)
broadcast_shapes: 10 passed, 0 failed out of 10
